# 🫁 Chest X-Ray Tiered Classifier — Tam Otomatik Colab Eğitim Notebook'u

Bu notebook **sıfırdan çalışır** ve her şeyi otomatik yapar:

1. GitHub repo'sunu clone eder (`xdboi123yes/xray`)
2. Bağımlılıkları kurar (Colab base image ile uyumlu, **otomatik kernel restart**)
3. Kaggle'dan NIH ChestX-ray14 dataset'ini indirir
4. Veri preprocess + split (train/val/test/calibration)
5. Ark+ checkpoint indirir (fallback: Swin-Base ImageNet)
6. Tier 1 (MobileNetV2) + Tier 2 (EffNetB4) + Tier 2 (Ark+) eğitir
7. Calibration (temperature scaling + ECE)
8. Ablation matrix (A1–A15) + dürüst `ablation.json`
9. İstatistiksel testler (DeLong + Bootstrap)
10. ONNX + INT8 export
11. Drive'a senkronla **ve** tek ZIP olarak indir

**Kullanım:** Üstten alta sırayla `Shift+Enter`. Tek manuel adım: Cell #2'de `kaggle.json` upload.

**GPU şart:** `Runtime > Change runtime type > T4 GPU` (veya A100/L4).

## 0. Konfigürasyon (tek yerden değiştir)

In [ ]:
# ============================================================================
# KONFİGÜRASYON — Tüm switch'ler burada
# ============================================================================
GITHUB_REPO_URL  = 'https://github.com/xdboi123yes/xray.git'
GITHUB_BRANCH    = 'main'
PROJECT_ROOT     = '/content/xray'                      # Clone hedefi
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/xray_outputs' # Output mirror
USE_DRIVE        = True   # False yaparsan Drive mount edilmez

# Eğitim switch'leri — herhangi birini False yaparsan o tier atlanır
RUN_TIER1         = True   # MobileNetV2
RUN_TIER2_EFFNET  = True   # EfficientNet-B4
RUN_TIER2_ARK     = True   # Ark+ Swin
RUN_CALIBRATION   = True
RUN_ABLATION      = True
RUN_STAT_TESTS    = True
RUN_ONNX_EXPORT   = True

# Hızlı dry-run modu (1 epoch / tier, validasyon için)
DRY_RUN = False

# Kaggle dataset slug — NIH ChestX-ray14 (Kaggle mirror)
KAGGLE_DATASET = 'nih-chest-xrays/data'

print('Konfigürasyon yüklendi.')
print(f'  Repo:   {GITHUB_REPO_URL} ({GITHUB_BRANCH})')
print(f'  Train:  Tier1={RUN_TIER1}  Tier2-Eff={RUN_TIER2_EFFNET}  Tier2-Ark={RUN_TIER2_ARK}')
print(f'  Drive:  {USE_DRIVE} → {DRIVE_OUTPUT_DIR}')
print(f'  DryRun: {DRY_RUN}')

## 1. Kaggle credentials upload (tek seferlik manuel)

`kaggle.json` dosyanı (Kaggle Account > API > Create New API Token) sürükle-bırak.

In [ ]:
import os, json
from pathlib import Path

KAGGLE_CONFIG_DIR = Path.home() / '.kaggle'
KAGGLE_CONFIG_DIR.mkdir(exist_ok=True)
KAGGLE_JSON_PATH = KAGGLE_CONFIG_DIR / 'kaggle.json'

if KAGGLE_JSON_PATH.exists():
    print(f'✅ Kaggle credentials zaten var: {KAGGLE_JSON_PATH}')
else:
    from google.colab import files
    print('📤 kaggle.json dosyanı yükle (sürükle-bırak):')
    uploaded = files.upload()
    fname = next(iter(uploaded.keys()))
    if fname != 'kaggle.json':
        # İsim farklıysa rename et
        os.rename(fname, 'kaggle.json')
        fname = 'kaggle.json'
    Path(fname).rename(KAGGLE_JSON_PATH)
    KAGGLE_JSON_PATH.chmod(0o600)
    # Sanity check
    with open(KAGGLE_JSON_PATH) as f:
        creds = json.load(f)
    assert 'username' in creds and 'key' in creds, 'kaggle.json bozuk'
    print(f'✅ Kaggle credentials kuruldu: user={creds["username"]}')

## 2. Drive mount (output sync için)

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print(f'✅ Drive mounted. Output mirror: {DRIVE_OUTPUT_DIR}')
else:
    print('⏭️  Drive atlandı (USE_DRIVE=False).')

## 3. GitHub'dan clone

In [ ]:
import subprocess

if os.path.isdir(PROJECT_ROOT):
    print(f'📂 Repo zaten var: {PROJECT_ROOT}. Pull ediyorum...')
    subprocess.check_call(['git', '-C', PROJECT_ROOT, 'fetch', 'origin'])
    subprocess.check_call(['git', '-C', PROJECT_ROOT, 'checkout', GITHUB_BRANCH])
    subprocess.check_call(['git', '-C', PROJECT_ROOT, 'reset', '--hard', f'origin/{GITHUB_BRANCH}'])
else:
    subprocess.check_call(['git', 'clone', '--branch', GITHUB_BRANCH, '--depth', '1', GITHUB_REPO_URL, PROJECT_ROOT])
    print(f'✅ Clone tamamlandı: {PROJECT_ROOT}')

os.chdir(PROJECT_ROOT)
print(f'📍 cwd: {os.getcwd()}')
print()
print(subprocess.check_output(['git', 'log', '-1', '--oneline']).decode())

## 4. Bağımlılıkları kur (Colab base ile uyumlu) + otomatik kernel restart

**ÖNEMLİ:** Bu hücre install bitince kernel'i restart eder. Restart sonrası bir sonraki hücreden devam et.
İlk run'da `flag` dosyası yoksa kurulum yapar, ikinci run'da pas geçer.

In [ ]:
import os, sys, subprocess

INSTALL_FLAG = '/content/.deps_installed'

if os.path.exists(INSTALL_FLAG):
    print('✅ Bağımlılıklar zaten kurulu (flag mevcut). Atlıyorum.')
else:
    print('📦 Bağımlılıklar kuruluyor (3-5 dakika)...')
    # Önce Colab base image ile çakışan paketleri pin'le
    # requirements.txt'teki versiyonları zorla ama numpy gibi sistemsel olanları upgrade et
    
    # 1) Önce Colab'ın eski tqdm/protobuf/numpy'sini training-uyumlu sürümlere getir
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--quiet',
        '--upgrade', 'tqdm>=4.66.3', 'packaging>=24.0'
    ])
    
    # 2) Repo requirements'ını kur
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--quiet',
        '-r', 'requirements.txt'
    ])
    
    # 3) Training-ekstraları (varsa)
    if os.path.exists('requirements-training.txt'):
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '--quiet',
            '-r', 'requirements-training.txt'
        ])
    
    # 4) Colab-spesifik ekstra
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--quiet',
        'opencv-python-headless', 'kaggle'
    ])
    
    # Flag'i bırak ki restart sonrası tekrar kurmayalım
    Path(INSTALL_FLAG).touch()
    print('✅ Tüm bağımlılıklar kuruldu.')
    print()
    print('🔄 Kernel restart ediliyor — bu beklenen davranış.')
    print('   Restart sonrası bir sonraki hücreden DEVAM ET (üstteki hücreleri yeniden çalıştırma).')
    print()
    # Otomatik restart
    import IPython
    IPython.Application.instance().kernel.do_shutdown(restart=True)

## 5. Post-restart: konfigürasyonu ve cwd'yi geri yükle

Kernel restart'tan sonra bu hücreden devam et.

In [ ]:
import os, sys, subprocess
from pathlib import Path

# Konfigürasyon değişkenlerini yeniden yükle (kernel restart kaybı)
GITHUB_REPO_URL  = 'https://github.com/xdboi123yes/xray.git'
GITHUB_BRANCH    = 'main'
PROJECT_ROOT     = '/content/xray'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/xray_outputs'
USE_DRIVE        = True
RUN_TIER1         = True
RUN_TIER2_EFFNET  = True
RUN_TIER2_ARK     = True
RUN_CALIBRATION   = True
RUN_ABLATION      = True
RUN_STAT_TESTS    = True
RUN_ONNX_EXPORT   = True
DRY_RUN = False
KAGGLE_DATASET = 'nih-chest-xrays/data'

os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Drive remount (restart sonrası kaybolur)
if USE_DRIVE and not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

import torch
print(f'📍 cwd: {os.getcwd()}')
print(f'🐍 Python: {sys.version.split()[0]}')
print(f'🔥 PyTorch: {torch.__version__}')
print(f'⚡ CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('   ⚠️  GPU yok! Runtime > Change runtime type > T4 GPU')

## 6. NIH Dataset'ini Kaggle'dan indir ve birleştir

**Boyut:** ~45 GB indirme + extract. Drive'da `MyDrive/nih_data/images/` varsa direkt oradan kullanır.

In [ ]:
import os, shutil, glob, subprocess

DRIVE_DATA   = '/content/drive/MyDrive/nih_data'
COLAB_DATA   = '/content/nih_data'
MERGED_DIR   = f'{COLAB_DATA}/images'        # flat folder, Colab disk
DRIVE_MERGED = f'{DRIVE_DATA}/images'         # mirror in Drive
CSV_NAME     = 'Data_Entry_2017.csv'
MIN_IMAGES   = 100000

os.makedirs(COLAB_DATA, exist_ok=True)

def count_files(d):
    if not os.path.isdir(d): return 0
    return len([f for f in os.listdir(d) if os.path.isfile(os.path.join(d, f))])

def merge_subdirs(base_dir, target_dir):
    """images_001..012/images/ → tek flat target_dir"""
    os.makedirs(target_dir, exist_ok=True)
    for i in range(1, 13):
        sub = os.path.join(base_dir, f'images_{i:03d}', 'images')
        if not os.path.isdir(sub):
            continue
        files = os.listdir(sub)
        print(f'  📦 images_{i:03d}/images/ → {len(files)} dosya')
        for fname in files:
            src, dst = os.path.join(sub, fname), os.path.join(target_dir, fname)
            if not os.path.exists(dst):
                shutil.move(src, dst)
    for i in range(1, 13):
        sub = os.path.join(base_dir, f'images_{i:03d}')
        if os.path.isdir(sub):
            shutil.rmtree(sub, ignore_errors=True)
    print(f'  ✅ Birleştirme: {count_files(target_dir)} dosya')

# Strateji: a) Colab disk b) Drive merged c) Drive raw d) Kaggle download
n_colab = count_files(MERGED_DIR)
if n_colab >= MIN_IMAGES:
    print(f'✅ (a) Colab disk hazır: {n_colab} dosya')
elif count_files(DRIVE_MERGED) >= MIN_IMAGES:
    print(f'📂 (b) Drive merged → Colab kopyalanıyor (~5 dk)...')
    os.makedirs(MERGED_DIR, exist_ok=True)
    for src_file in glob.glob(os.path.join(DRIVE_MERGED, '*')):
        dst_file = os.path.join(MERGED_DIR, os.path.basename(src_file))
        if not os.path.exists(dst_file):
            shutil.copy2(src_file, dst_file)
    print(f'   ✅ {count_files(MERGED_DIR)} dosya')
elif any(os.path.isdir(os.path.join(COLAB_DATA, f'images_{i:03d}')) for i in range(1, 13)):
    print('📦 (c) Colab raw subdirs → merge')
    merge_subdirs(COLAB_DATA, MERGED_DIR)
else:
    print(f'🌐 (d) Kaggle\'dan indiriyorum: {KAGGLE_DATASET}')
    print('   ~45 GB — 10-20 dk sürebilir.')
    os.environ['KAGGLE_CONFIG_DIR'] = str(Path.home() / '.kaggle')
    subprocess.check_call([
        'kaggle', 'datasets', 'download',
        '-d', KAGGLE_DATASET,
        '-p', COLAB_DATA,
        '--unzip'
    ])
    # Kaggle indirmesi: COLAB_DATA/images_001..012/images/*.png
    print('🔧 Subdirleri birleştiriyorum...')
    merge_subdirs(COLAB_DATA, MERGED_DIR)

# CSV'yi data/raw/'a kopyala
csv_src_candidates = [
    os.path.join(COLAB_DATA, CSV_NAME),
    os.path.join(DRIVE_DATA, CSV_NAME),
]
csv_dst = os.path.join(PROJECT_ROOT, 'data', 'raw', CSV_NAME)
os.makedirs(os.path.dirname(csv_dst), exist_ok=True)
for csv_src in csv_src_candidates:
    if os.path.exists(csv_src):
        shutil.copy2(csv_src, csv_dst)
        print(f'✅ CSV: {csv_src} → {csv_dst}')
        break
else:
    raise FileNotFoundError(f'{CSV_NAME} bulunamadı. Kaggle download eksik.')

IMAGE_DIR = MERGED_DIR
print(f'\n📊 Final: {count_files(IMAGE_DIR):,} görüntü @ {IMAGE_DIR}')

# IMAGE_DIR'i preprocess için bir text'e yaz
os.makedirs(os.path.join(PROJECT_ROOT, 'data', 'processed'), exist_ok=True)
with open(os.path.join(PROJECT_ROOT, 'data', 'processed', 'image_dir.txt'), 'w') as f:
    f.write(IMAGE_DIR)

## 7. Veri preprocess + train/val/test/calibration split

In [ ]:
import pandas as pd

# preprocess.py train/val/test üretir, calibration set yoksa val'den ayırırız
subprocess.check_call([
    sys.executable, 'scripts/preprocess.py',
    '--image-dir', IMAGE_DIR,
])

print('\n📊 Split istatistikleri:')
for split in ['train', 'val', 'test']:
    p = f'data/processed/{split}.csv'
    if os.path.exists(p):
        df = pd.read_csv(p)
        label_col = 'Label' if 'Label' in df.columns else 'label'
        n_pos = (df[label_col] == 1).sum()
        n_neg = (df[label_col] == 0).sum()
        print(f'  {split:5s}: {len(df):6d} | Pneumo: {n_pos:5d} | Normal: {n_neg:5d} | 1:{n_neg/max(n_pos,1):.1f}')
    else:
        print(f'  ⚠️  {p} eksik!')

# Calibration split (val'in %30'u) — calibration_service için lazım
calib_csv = 'data/processed/calibration.csv'
if not os.path.exists(calib_csv):
    val_df = pd.read_csv('data/processed/val.csv')
    calib_df = val_df.sample(frac=0.3, random_state=42)
    calib_df.to_csv(calib_csv, index=False)
    print(f'  ✅ Calibration split oluşturuldu: {len(calib_df)} sample')

## 8. Ark+ checkpoint indir (fallback: Swin-Base ImageNet)

In [ ]:
if RUN_TIER2_ARK:
    print('⬇️  Ark+ checkpoint indiriliyor (4 URL deneme + Swin-Base fallback)...')
    rc = subprocess.call([sys.executable, 'scripts/download_ark_plus.py'])
    print(f'   Exit code: {rc}')
    print()
    subprocess.call(['ls', '-la', 'outputs/models/'])
else:
    print('⏭️  Ark+ atlandı (RUN_TIER2_ARK=False).')

## 9. Eğitim — Tier 1 (MobileNetV2)

Output: `outputs/models/Tier1_MobileNetV2_Colab/best_model.pth`

In [ ]:
from application.dto.training_config_dto import TrainingConfigDTO
from application.services.training_service import TrainingService
import shutil

service = TrainingService()

if RUN_TIER1:
    tier1_cfg = TrainingConfigDTO(
        backbone='mobilenet_v2',
        run_name='Tier1_MobileNetV2_Colab',
        epochs=1 if DRY_RUN else 50,
        batch_size=32,
        lr_backbone=1e-4,
        lr_head=1e-3,
        early_stopping_patience=7,
        seed=42,
        use_synthetic=False,  # Synthetic kullanmıyoruz
    )
    print(f'🚀 Tier 1 eğitim başlıyor (epochs={tier1_cfg.epochs})...')
    tier1_result = service.train_model(tier1_cfg, dry_run=DRY_RUN)
    print(f'\n✅ Tier 1 tamam:')
    print(f'   Best Val AUC: {tier1_result["best_val_auc"]:.4f}')
    print(f'   Weights: {tier1_result["model_weight_path"]}')
    
    # export_onnx.py'nin beklediği isime symlink/copy
    legacy_path = 'outputs/models/best_tier1_mobilenet.pth'
    shutil.copy2(tier1_result['model_weight_path'], legacy_path)
    print(f'   Legacy alias: {legacy_path}')
else:
    print('⏭️  Tier 1 atlandı.')

## 10. Eğitim — Tier 2 (EfficientNet-B4)

Output: `outputs/models/Tier2_EfficientNetB4_Colab/best_model.pth`

In [ ]:
if RUN_TIER2_EFFNET:
    tier2_eff_cfg = TrainingConfigDTO(
        backbone='efficientnet_b4',
        run_name='Tier2_EfficientNetB4_Colab',
        epochs=1 if DRY_RUN else 50,
        batch_size=16,
        lr_backbone=5e-5,
        lr_head=5e-4,
        early_stopping_patience=7,
        seed=42,
        use_synthetic=False,
    )
    print(f'🚀 Tier 2 EffNetB4 eğitim başlıyor (epochs={tier2_eff_cfg.epochs})...')
    tier2_eff_result = service.train_model(tier2_eff_cfg, dry_run=DRY_RUN)
    print(f'\n✅ Tier 2 EffNet tamam:')
    print(f'   Best Val AUC: {tier2_eff_result["best_val_auc"]:.4f}')
    print(f'   Weights: {tier2_eff_result["model_weight_path"]}')
    
    legacy_path = 'outputs/models/best_tier2_efficientnet.pth'
    shutil.copy2(tier2_eff_result['model_weight_path'], legacy_path)
    print(f'   Legacy alias: {legacy_path}')
else:
    print('⏭️  Tier 2 EffNet atlandı.')

## 11. Eğitim — Tier 2 (Ark+ Swin)

Output: `outputs/models/Tier2_ArkPlus_Colab/best_model.pth`

In [ ]:
if RUN_TIER2_ARK:
    tier2_ark_cfg = TrainingConfigDTO(
        backbone='ark_plus',
        run_name='Tier2_ArkPlus_Colab',
        epochs=1 if DRY_RUN else 40,
        batch_size=12,
        lr_backbone=2e-5,
        lr_head=2e-4,
        early_stopping_patience=8,
        seed=42,
        use_synthetic=False,
    )
    print(f'🚀 Tier 2 Ark+ eğitim başlıyor (epochs={tier2_ark_cfg.epochs})...')
    tier2_ark_result = service.train_model(tier2_ark_cfg, dry_run=DRY_RUN)
    print(f'\n✅ Tier 2 Ark+ tamam:')
    print(f'   Best Val AUC: {tier2_ark_result["best_val_auc"]:.4f}')
    print(f'   Weights: {tier2_ark_result["model_weight_path"]}')
    
    legacy_path = 'outputs/models/best_tier2_arkplus.pth'
    shutil.copy2(tier2_ark_result['model_weight_path'], legacy_path)
    print(f'   Legacy alias: {legacy_path}')
else:
    print('⏭️  Tier 2 Ark+ atlandı.')

## 12. Calibration — Temperature Scaling + ECE

Tier 2 EffNet üzerinde temperature scaling, validation set'te ECE hesabı.

In [ ]:
if RUN_CALIBRATION:
    import torch
    import numpy as np
    from torch.utils.data import DataLoader
    from application.services.calibration_service import CalibrationService
    from core.models.factory import ModelFactory
    from infrastructure.data.dataset import NIHChestXrayDataset
    import core.models.tier2_efficientnet  # registration
    
    weights_path = 'outputs/models/best_tier2_efficientnet.pth'
    if not os.path.exists(weights_path):
        print(f'⚠️  {weights_path} yok, calibration atlanıyor.')
    else:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model = ModelFactory.create('efficientnet_b4')
        state = torch.load(weights_path, map_location=device)
        # checkpoint observer state_dict or direct weights
        if isinstance(state, dict) and 'model_state_dict' in state:
            model.load_state_dict(state['model_state_dict'])
        else:
            model.load_state_dict(state)
        model.to(device).eval()
        
        calib_ds = NIHChestXrayDataset(csv_file='data/processed/calibration.csv')
        calib_loader = DataLoader(calib_ds, batch_size=32, shuffle=False, num_workers=0)
        
        calib_service = CalibrationService()
        T = calib_service.calibrate_model(model, calib_loader, device)
        
        # ECE hesabı
        probs, labels = [], []
        with torch.no_grad():
            for batch in calib_loader:
                if isinstance(batch, (list, tuple)):
                    x, y = batch[0].to(device), batch[1]
                else:
                    x, y = batch['image'].to(device), batch['label']
                logits = model(x) / T
                p = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
                probs.append(p)
                labels.append(y.numpy() if hasattr(y, 'numpy') else np.array(y))
        probs = np.concatenate(probs)
        labels = np.concatenate(labels)
        ece = calib_service.calculate_ece(probs, labels, n_bins=10)
        
        # Save temperature
        os.makedirs('outputs/results', exist_ok=True)
        torch.save({'temperature': T, 'ece': ece}, 'outputs/results/temperature.pt')
        
        # Reliability diagram
        calib_service.save_reliability_diagram(probs, labels, filename='reliability_tier2_eff.png')
        
        print(f'✅ Calibration:')
        print(f'   Optimal T: {T:.4f}')
        print(f'   ECE @ T:   {ece:.4f}')
else:
    print('⏭️  Calibration atlandı.')

## 13. Ablation Matrix (A1–A15)

In [ ]:
if RUN_ABLATION:
    from application.orchestration.ablation_runner import AblationRunner
    runner = AblationRunner()
    print(f'🔬 Ablation runs (dry_run={DRY_RUN})...')
    results = runner.run_all(dry_run=DRY_RUN)
    print(f'\n✅ Ablation: {len(results)} run tamamlandı.')
    
    # Dürüst ablation.json üret
    print('\n📝 ablation.json regenerating from MLflow...')
    subprocess.check_call([sys.executable, 'scripts/build_ablation_json.py'])
    
    import json
    with open('outputs/results/ablation.json') as f:
        ab_data = json.load(f)
    real = sum(1 for r in ab_data if r.get('provenance') == 'mlflow_run')
    print(f'   Real (mlflow_run): {real}/{len(ab_data)}')
else:
    print('⏭️  Ablation atlandı.')

## 14. İstatistiksel Testler (DeLong + Bootstrap)

In [ ]:
if RUN_STAT_TESTS:
    subprocess.check_call([
        sys.executable, 'scripts/statistical_tests.py',
        '--output', 'outputs/results/statistical_comparison.csv',
    ])
    
    if os.path.exists('outputs/results/statistical_comparison.csv'):
        df = pd.read_csv('outputs/results/statistical_comparison.csv')
        print(f'\n✅ Statistical tests: {len(df)} comparison')
        print(df.head(10).to_string())
else:
    print('⏭️  Stat tests atlandı.')

## 15. ONNX + INT8 export (mobile-friendly)

In [ ]:
if RUN_ONNX_EXPORT:
    for model_name, weights in [
        ('tier1', 'outputs/models/best_tier1_mobilenet.pth'),
        ('tier2', 'outputs/models/best_tier2_efficientnet.pth'),
    ]:
        if not os.path.exists(weights):
            print(f'⚠️  {weights} yok, {model_name} ONNX export atlandı.')
            continue
        
        cmd = [sys.executable, 'scripts/export_onnx.py', '--model', model_name]
        if model_name == 'tier2':
            cmd.append('--quantize')
        print(f'📦 ONNX export: {" ".join(cmd)}')
        subprocess.check_call(cmd)
    
    print()
    subprocess.call(['ls', '-la', 'outputs/models/'])
else:
    print('⏭️  ONNX export atlandı.')

## 16. Drive'a senkronla + ZIP indir

In [ ]:
import shutil, datetime

TIMESTAMP = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
ZIP_NAME  = f'xray_outputs_{TIMESTAMP}.zip'
ZIP_PATH  = f'/content/{ZIP_NAME}'

# Drive sync
if USE_DRIVE and os.path.exists('/content/drive/MyDrive'):
    print(f'📤 Drive sync → {DRIVE_OUTPUT_DIR}/{TIMESTAMP}/')
    run_dir = os.path.join(DRIVE_OUTPUT_DIR, TIMESTAMP)
    os.makedirs(run_dir, exist_ok=True)
    for subdir in ['outputs', 'experiments']:
        src = os.path.join(PROJECT_ROOT, subdir)
        if os.path.exists(src):
            dst = os.path.join(run_dir, subdir)
            print(f'   {src} → {dst}')
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
    print('   ✅ Drive senkron tamam.')

# ZIP oluştur
print(f'\n📦 ZIP oluşturuluyor: {ZIP_NAME}')
shutil.make_archive(
    base_name=ZIP_PATH.replace('.zip', ''),
    format='zip',
    root_dir=PROJECT_ROOT,
    base_dir='outputs',
)
size_mb = os.path.getsize(ZIP_PATH) / 1e6
print(f'   ✅ {ZIP_PATH} ({size_mb:.1f} MB)')

# ZIP'i indir
from google.colab import files
print(f'\n⬇️  Tarayıcıya indiriliyor...')
files.download(ZIP_PATH)
print('   ✅ İndirme başladı.')

## 17. Son sanity check

In [ ]:
print('📁 outputs/models/')
subprocess.call(['ls', '-la', 'outputs/models/'])
print('\n📁 outputs/results/')
subprocess.call(['ls', '-la', 'outputs/results/'])
print('\n💾 Disk usage:')
subprocess.call(['du', '-sh', 'outputs/', 'experiments/mlruns/'], stderr=subprocess.DEVNULL)
print('\n🎉 Tamam kanka, bitti.')